# 저장 ML + DL 확률 결합·평가·개인 계정 MLflow 기록

`fusion.yaml`에서 저장된 ML·DL artifact를 선택합니다. 기반 모델을 고정한 채 validation을 평가하고, `run_final_test=True`이면 같은 가중치로 test를 평가합니다. 각 결과는 `mlflow_logging=True`일 때 각 PC의 개인 계정으로 기록합니다. Raw DL artifact도 지원하며, 동일 PID의 split·SCD 라벨과 raw 전처리 설정을 검증합니다.

In [ ]:
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import yaml

repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/fusion.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 내부에서 노트북을 실행하세요")
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

In [ ]:
from Members.sang.model.config import load_yaml, resolve_config_path
from Members.sang.model.evaluation.evaluation import evaluate_saved_model
from Members.sang.model.tracking.check_connection import check_mlflow_connection

mlflow_logging = True
run_final_test = True  # 모델/가중치를 비교하는 중에는 False
fusion_config = repo / "Members/sang/model/fusion.yaml"
if mlflow_logging:
    check_mlflow_connection()

## Validation 평가 후 가중치를 고정해 test 평가

수동 가중치는 음수가 아니며 합이 1이어야 합니다. `mode: learn`이면 validation만으로 가중치를 학습합니다. 아래 test 단계는 validation run의 저장 가중치를 재사용합니다. 임시 설정을 사용하므로 원본 YAML을 바꾸지 않습니다. 환자별 예측은 로컬에 보관하고 집계 평가 결과를 MLflow에 기록합니다.

In [ ]:
fusion = load_yaml(fusion_config)
fusion["basic_config"] = str(resolve_config_path(fusion["basic_config"], fusion_config))
for modality in ("ml", "dl"):
    fusion[modality]["artifact_dir"] = str(
        resolve_config_path(fusion[modality]["artifact_dir"], fusion_config)
    )
if fusion["fusion"].get("saved_run_dir"):
    fusion["fusion"]["saved_run_dir"] = str(
        resolve_config_path(fusion["fusion"]["saved_run_dir"], fusion_config)
    )
eval_path = repo / "Members/sang/model/evaluation/eval.yaml"
evaluation = load_yaml(eval_path)
evaluation["basic_config"] = fusion["basic_config"]
evaluation["evaluation"].update(
    target="fusion",
    mlflow_logging=mlflow_logging,
    threshold_override=None,
    results_directory=str(
        resolve_config_path(evaluation["evaluation"]["results_directory"], eval_path)
    ),
)
with TemporaryDirectory(prefix="fusion-evaluation-") as temporary:
    temporary = Path(temporary)
    temporary_fusion = temporary / "fusion.yaml"
    temporary_evaluation = temporary / "eval.yaml"
    evaluation["evaluation"]["fusion_config"] = str(temporary_fusion)
    temporary_fusion.write_text(yaml.safe_dump(fusion), encoding="utf-8")
    temporary_evaluation.write_text(yaml.safe_dump(evaluation), encoding="utf-8")
    fusion_run = evaluate_saved_model(temporary_evaluation, split_override="validation")
    if run_final_test:
        fusion["fusion"].update(mode="saved", saved_run_dir=str(fusion_run))
        temporary_fusion.write_text(yaml.safe_dump(fusion), encoding="utf-8")
        test_run = evaluate_saved_model(temporary_evaluation, split_override="test")

print("Validation:", fusion_run)
print(json.loads((fusion_run / "metrics.json").read_text()))
display(pd.read_csv(fusion_run / "validation_predictions.csv").head())
if run_final_test:
    print("Test:", test_run)
    print(json.loads((test_run / "metrics.json").read_text()))

## 전송 재시도

전송 실패 시 로컬 결과는 그대로 남고 `mlflow_status.json`이 `PENDING`입니다. `log_evaluation_to_mlflow(fusion_run)` 또는 `log_evaluation_to_mlflow(test_run)`으로 평가를 반복하지 않고 전송만 재시도합니다. 함수는 `Members.sang.model.tracking.log_evaluation`에서 import합니다.